In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

from sqlalchemy import select, func
from tqdm.notebook import tqdm

from wordfreq import tokenize, word_frequency
from collections import Counter

import sys
from types import ModuleType

# fix for missing CPU hdbscan module
mock_hdbscan = ModuleType("hdbscan")
mock_hdbscan.HDBSCAN = type("HDBSCAN", (), {})
sys.modules["hdbscan"] = mock_hdbscan

from telegram_data_models import Message, Chat, MessageTextContent, ChatLanguage

from telegram_quality_control.db import get_conn_string

from telegram_quality_control.cleaning import clean_text
from telegram_quality_control.topics import Embeddings, Topics

In [ ]:
conn_str = get_conn_string(".env")

# Download chat info
sql = (
    select(Chat.id, Chat.name, Chat.description, ChatLanguage.lang)
    .join(ChatLanguage, ChatLanguage.chat_id == Chat.id)
    .order_by(Chat.id)
    .limit(100)
)
chat_df = pd.read_sql(sql, conn_str)
chat_df = chat_df.set_index("id")

chat_ids = list(chat_df.index)
chat_ids.sort()

chat_df

In [ ]:
def get_keywords(text, lang):
    text = text.lower()

    if lang == "ru":
        # remove latin letters
        text = re.sub(r'http\S+|#\S+|[a-zA-Z]+', '', text)

    text = clean_text(text)

    tokens = tokenize(text, lang)

    counts = Counter(tokens)
    total = sum(counts.values())

    scored = {
        word: (count / total) / np.log(word_frequency(word, lang) + 1e-9)
        for word, count in counts.items()
        if count > 2  # filter rare noise
    }

    top_keywords = sorted(scored, key=scored.get, reverse=True)[:50]
    return top_keywords


chat_df["keywords"] = ""

chat_sample_text = {}

for chat_id in tqdm(chat_ids):
    # Download messages
    sql = (
        select(
            MessageTextContent.message_id,
            Message.chat_id,
            func.coalesce(MessageTextContent.text, MessageTextContent.caption).label('content'),
        )
        .join(Message, Message.id == MessageTextContent.message_id)
        .where(Message.chat_id == chat_id)
    )

    lang = chat_df.loc[chat_id, "lang"]

    message_df = pd.read_sql(sql, conn_str)
    message_df = message_df[message_df["content"].notna()]
    if len(message_df) > 100:
        message_df = message_df.sample(100)
    message_text = " ".join(message_df["content"].astype(str))
    chat_sample_text[chat_id] = message_text

In [ ]:
def chunk_documents(chat_sample_text, snippet_length=500):
    snippets = []
    chat_ids = []

    for doc_id, text in chat_sample_text.items():
        chunks = [text[i : i + snippet_length] for i in range(0, len(text), snippet_length)]
        snippets.extend(chunks)
        chat_ids.extend([doc_id] * len(chunks))

    return snippets, chat_ids


docs, chat_ids = chunk_documents(chat_sample_text, snippet_length=200)

In [ ]:
embedding_model = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
embedding_cache = Embeddings(folder_tag="seed_chats", embedding_model=embedding_model)
embeddings = embedding_cache.create(docs)

In [ ]:
def get_chat_topic(chat_ids, topic_ids):
    # get the most characteristic topic per chat (the most overrepresented topic)
    unique_docs = list(dict.fromkeys(chat_ids))
    unique_topics = list(dict.fromkeys(topic_ids))
    doc_idx = {d: i for i, d in enumerate(unique_docs)}
    topic_idx = {t: i for i, t in enumerate(unique_topics)}

    # Build TF matrix (docs x topics)
    tf = np.zeros((len(unique_docs), len(unique_topics)))
    for doc_id, topic in zip(chat_ids, topic_ids):
        tf[doc_idx[doc_id], topic_idx[topic]] += 1

    # TF: normalize by doc length
    tf = tf / tf.sum(axis=1, keepdims=True)

    # IDF: log(n_docs / doc frequency per topic)
    doc_freq = (tf > 0).sum(axis=0)
    idf = np.log(len(unique_docs) / doc_freq)

    tfidf = tf * idf

    best_topic_indices = np.argmax(tfidf, axis=1)
    return {doc_id: unique_topics[i] for doc_id, i in zip(unique_docs, best_topic_indices)}

In [ ]:
for i in [1, 2, 5, 10]:
    print(f"Topic analysis with min_samples={i}")
    topic_cache = Topics(
        "seed_chats", min_samples=i, min_cluster_size=i, calculate_probabilities=False
    )
    topic_model, topics, probs = topic_cache.create(
        docs, embeddings, embedding_cache.embedding_model
    )

    topic_labels = topic_model.get_topic_info()[["Topic", "Name"]]
    topic_labels = topic_labels.set_index("Topic")["Name"].to_dict()

    chat_topic_ids = get_chat_topic(chat_ids, topics)
    chat_topic_labels = pd.DataFrame(columns=["topic_label"])
    chat_topic_labels.index.name = "chat_id"
    for chat_id, topic_id in chat_topic_ids.items():
        label = topic_labels[topic_id]
        label = ' '.join(label.split('_')[1:])
        chat_topic_labels.loc[chat_id] = label

    chat_df_results = chat_df.join(chat_topic_labels)
    chat_df_results = chat_df_results.sort_index()
    chat_df_results[["name", "lang", "topic_label"]].to_csv(f"./data/seed_chat_topics_{i}.csv")